# Direction A atlas — run all events on Colab

Runs the full IC-optimization pipeline (`heatwave_ic.pipeline.run_event`) over every event config in `configs/`, in validation-first order (PNW → St. John's → new zones), and collects the cross-zone summary in `data/atlas_summary.csv`.

**Before running:** Runtime → Change runtime type → **GPU (A100 or L4, High-RAM)**.

**Session budgeting:** each event is ~75 Adam iterations over a 7–11-day unroll — budget roughly 1–2 events per Colab session. The runner is resume-aware: completed events are skipped, and with Drive persistence (cell 3) finished runs survive session death, so just re-run this notebook until the atlas is complete.

In [1]:
REPO_URL = "https://github.com/ieadoboe/heatwave-initial-conditions.git"

import shutil, sys
from pathlib import Path

name = Path(REPO_URL).stem
candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / name]
root = next((p for p in candidates if (p / "heatwave_ic").is_dir()), None)
if root is None and "google.colab" in sys.modules:
    shutil.rmtree(name, ignore_errors=True)   # clear stale/partial clones
    !git clone {REPO_URL}
    root = Path.cwd() / name
if root is None or not (root / "heatwave_ic").is_dir():
    raise RuntimeError("heatwave_ic/ not found — clone failed or package not pushed.")
%cd {root}
sys.path.insert(0, str(root))
if "google.colab" in sys.modules:
    %pip install -q -U neuralgcm dinosaur gcsfs optax tqdm pyyaml zarr

/content/heatwave-initial-conditions


In [2]:
# Persistence: completed runs sync to Drive after EVERY event, and are
# restored at the start of the next session. Set PERSIST = False to skip
# (outputs then die with the VM). IC zarrs are never persisted (tens of GB)
# — they rebuild automatically when a session needs one.
PERSIST = True
PERSIST_DIR = "/content/drive/MyDrive/heatwave_atlas"

if PERSIST and "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(PERSIST_DIR).mkdir(parents=True, exist_ok=True)
persist_flag = f"--persist-dir {PERSIST_DIR}" if PERSIST else ""

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Run the atlas

The default runs everything not yet completed. To run a chosen subset this session (recommended: 1–2 events), use the second form.

In [3]:
!python scripts/run_atlas.py {persist_flag}

# Subset form — e.g. just the PNW validation run:
# !python scripts/run_atlas.py {persist_flag} --configs configs/pnw_jun2021.yaml

# Force a redo of an event (ignores its existing run dir):
# !python scripts/run_atlas.py {persist_flag} --rerun --configs configs/pnw_jun2021.yaml

python3: can't open file '/content/heatwave-initial-conditions/scripts/run_atlas.py': [Errno 2] No such file or directory


In [4]:
import pandas as pd

summary = pd.read_csv("data/atlas_summary.csv")
display(summary)

print("\nFigures written to plots/:")
for p in sorted(Path("plots").glob("*_storyline.pdf")):
    print(" ", p)

FileNotFoundError: [Errno 2] No such file or directory: 'data/atlas_summary.csv'

## Reading the result

- **`storyline_gain_C`** is the headline number per event: how much hotter the optimized-IC storyline peaks vs the unperturbed forecast (W&DL's PNW benchmark: **+3.7 °C**). If `pnw_jun2021` doesn't land near that, fix the pipeline before interpreting the other zones.
- Per-event outputs are in `data/opt_runs/<run-name>/`: `optimized.nc` / `original.nc` trajectories, initial-state fields (`*_opt.npy`, `*_original.npy`) for the sensitivity-map analysis, `losses.npy`, `storyline.csv`.
- Cross-zone comparison of the optimal-perturbation *structure* (the actual Direction A science) starts from those saved state fields.